In [1]:
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'
!pip install roboflow


  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-h6bndwal
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-h6bndwal
  Resolved https://github.com/facebookresearch/detectron2.git to commit 1e3e13bbf607b54f62205c4c33922521822fb298
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.2/105.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.6 MB/s eta 0:00:00
  Created wheel for 

In [4]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API = userdata.get('ROBOFLOW_API')
rf = Roboflow(api_key=ROBOFLOW_API)
project = rf.workspace("udy").project("visdrone-qr0vn")
version = project.version(2)
dataset = version.download("coco")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Visdrone-2 in coco:: 100%|██████████| 8633/8633 [00:14<00:00, 611.33it/s]


In [ ]:
dataset.location

In [8]:
import os
from detectron2.data.datasets import register_coco_instances
from detectron2.config import get_cfg
from detectron2 import model_zoo

# 1. Khai báo 2 thư mục train và valid
data_dir = dataset.location
register_coco_instances("visdrone_train", {}, f"{data_dir}/train/_annotations.coco.json", f"{data_dir}/train")
register_coco_instances("visdrone_val", {}, f"{data_dir}/valid/_annotations.coco.json", f"{data_dir}/valid")
# 2. Tạo file Config cho Cascade R-CNN
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.MASK_ON = False  # Chỉ lấy Bounding Box
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml")
cfg.DATASETS.TRAIN = ("visdrone_train",)
cfg.DATASETS.TEST = ("visdrone_val",)
cfg.DATALOADER.NUM_WORKERS = 2
# Thông số học (Training)
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 2e-3
cfg.SOLVER.MAX_ITER = 50000
cfg.SOLVER.STEPS = (40000, 47500)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 10  # VisDrone có 10 nhóm

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print('config done')

config done


In [9]:
import detectron2.data.transforms as T
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader, DatasetMapper

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        # Đây là chỗ chứa Augmentation
        custom_augmentations = [
            T.ResizeShortestEdge(short_edge_length=(640, 672, 704, 736, 768, 800), max_size=1333, sample_style="choice"),
            T.RandomCrop("relative_range", (0.5, 0.5)), # Cắt ngẫu nhiên 50% ảnh
            T.RandomFlip(prob=0.5, horizontal=True, vertical=False), # Lật ngang
            T.RandomBrightness(0.8, 1.2), # Đổi độ sáng
            T.RandomContrast(0.8, 1.2), # Đổi độ tương phản
        ]
        mapper = DatasetMapper(cfg, is_train=True, augmentations=custom_augmentations)
        return build_detection_train_loader(cfg, mapper=mapper)


In [10]:
trainer = CustomTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

[10/06 09:37:22 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

roi_heads.box_predictor.0.cls_score.{bias, weight}
roi_heads.box_predictor.1.cls_score.{bias, weight}
roi_heads.box_predictor.2.cls_score.{bias, weight}
  roi_heads.mask_head.mask_fcn1.{bias, weight}
  roi_heads.mask_head.mask_fcn2.{bias, weight}
  roi_heads.mask_head.mask_fcn3.{bias, weight}
  roi_heads.mask_head.mask_fcn4.{bias, weight}
  roi_heads.mask_head.deconv.{bias, weight}
  roi_heads.mask_head.predictor.{bias, weight}


[10/06 09:37:26 d2.engine.train_loop]: Starting training from iteration 0
[10/06 09:37:49 d2.utils.events]:  eta: 15:15:23  iter: 19  total_loss: 9.905  loss_cls_stage0: 2.395  loss_box_reg_stage0: 0.4654  loss_cls_stage1: 2.357  loss_box_reg_stage1: 0.7477  loss_cls_stage2: 2.337  loss_box_reg_stage2: 0.6976  loss_rpn_cls: 0.4195  loss_rpn_loc: 0.398    time: 1.0857  last_time: 1.3424  data_time: 0.0489  last_data_time: 0.0156   lr: 3.9962e-05  max_mem: 5403M
[10/06 09:38:13 d2.utils.events]:  eta: 16:18:27  iter: 39  total_loss: 8.137  loss_cls_stage0: 1.925  loss_box_reg_stage0: 0.5309  loss_cls_stage1: 1.875  loss_box_reg_stage1: 0.8126  loss_cls_stage2: 1.835  loss_box_reg_stage2: 0.6266  loss_rpn_cls: 0.2436  loss_rpn_loc: 0.3478    time: 1.1640  last_time: 1.2956  data_time: 0.0196  last_data_time: 0.0135   lr: 7.9922e-05  max_mem: 5534M
[10/06 09:38:37 d2.utils.events]:  eta: 16:44:17  iter: 59  total_loss: 5.302  loss_cls_stage0: 1.054  loss_box_reg_stage0: 0.432  loss_cls_sta

KeyboardInterrupt: 

In [ ]:
!pip install huggingface_hub

import os
from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get('HF_TOKEN')
api = HfApi(token=hf_token)

repo_id = "huynoob1312/detectron2-cascade-rcnn-visdrone"
api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")

model_path = "./output/model_final.pth"
if os.path.exists(model_path):
    api.upload_file(
        path_or_fileobj=model_path,
        path_in_repo="model_final.pth",
        repo_id=repo_id,
    )
else:
    print(f"❌ Lỗi: Không tìm thấy file {model_path}!")

for file_name in ["config.yaml", "metrics.json"]:
    file_path = f"./output/{file_name}"
    if os.path.exists(file_path):
        print(f"Đang đẩy file {file_name}...")
        api.upload_file(
            path_or_fileobj=file_path,
            path_in_repo=file_name,
            repo_id=repo_id,
        )